# Módulo 06 · Aula 03 — Arquitetura e Banco de Dados

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O `main.py` da API está com 900 linhas. Ontem eu mudei o cálculo de margem e quebrei a rota de pedidos — que não tem nada a ver. E os dados ainda estão num dicionário: toda vez que a gente reinicia o servidor, some tudo."*
> — Você, para você mesmo

Duas dores:

1. **Tudo num arquivo só.** Você já resolveu isso no M04 com camadas. Falta aplicar aqui.
2. **Os dados não persistem.** Você já tem PostgreSQL e SQLAlchemy do M05. Falta conectar.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Estrutura de projeto | Onde cada coisa mora |
| 2 | `APIRouter` | Quebrar a app em módulos |
| 3 | **`Depends`** | 🎯 O coração do FastAPI |
| 4 | **Sessão por requisição** | 🔴 O erro que derruba produção |
| 5 | CRUD com banco de verdade | Transação, commit, rollback |
| 6 | `lifespan` | Ligar e desligar recursos |
| 7 | Camadas | Rota → serviço → repositório |

> 💭 **Esta aula costura tudo.** As camadas do M04, o SQLAlchemy do M05 e as rotas do M06 viram um sistema só.

## ⚙️ Preparação

Vamos escrever **arquivos de verdade** com `%%writefile` e importá-los — porque estrutura de projeto não se aprende com tudo numa célula.

Usamos SQLite via SQLAlchemy: o mesmo código roda em PostgreSQL trocando a URL.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 06 · Aula 03
# ═══════════════════════════════════════════════════════════════
import json
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("sqlalchemy", "sqlalchemy"), ("pydantic", "pydantic")]:
    _garantir(_p, _m)

import fastapi
import sqlalchemy
from fastapi.testclient import TestClient

print(f"✅ FastAPI {fastapi.__version__}   SQLAlchemy {sqlalchemy.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Pasta de trabalho: aqui nasce o projeto desta aula
# ═══════════════════════════════════════════════════════════════
BASE = Path("aula_06_03").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
(BASE / "app" / "rotas").mkdir(parents=True)

# 🔑 Deixa a pasta importável (é o que o `pip install -e .` faria)
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

print(f"📁 {BASE}")


# ═══════════════════════════════════════════════════════════════
#  Auxiliares
# ═══════════════════════════════════════════════════════════════

def req(cliente, metodo: str, caminho: str, mostrar_corpo=True, **kwargs):
    """Faz uma requisição e imprime o resultado de forma legível."""
    resposta = getattr(cliente, metodo.lower())(caminho, **kwargs)
    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    print(f"{cor} {metodo.upper():<7} {caminho:<42} → {resposta.status_code}")
    if kwargs.get("json") is not None:
        corpo = json.dumps(kwargs["json"], ensure_ascii=False)
        print(f"   envio  : {corpo[:130]}{'...' if len(corpo) > 130 else ''}")
    if mostrar_corpo:
        try:
            texto = json.dumps(resposta.json(), ensure_ascii=False, indent=2)
            linhas = texto.splitlines()
            for linha in linhas[:14]:
                print(f"   {linha}")
            if len(linhas) > 14:
                print(f"   ... (+{len(linhas) - 14} linhas)")
        except Exception:
            if resposta.text.strip():
                print(f"   {resposta.text[:200]}")
    print()
    return resposta


def arvore(raiz: Path, prefixo=""):
    """Imprime a estrutura de pastas."""
    itens = sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
    itens = [i for i in itens if i.name != "__pycache__"]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        print(f"{prefixo}{'└── ' if ultimo else '├── '}{item.name}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


def recarregar(*modulos):
    """Remove módulos do cache para que %%writefile tenha efeito."""
    for nome in list(sys.modules):
        if any(nome == m or nome.startswith(m + ".") for m in modulos):
            del sys.modules[nome]


print("✅ `req()`, `arvore()` e `recarregar()` prontos")

## 1. Onde cada coisa mora

Um `main.py` de 900 linhas não é um problema de estética. É um problema de **acoplamento**: mudar o cálculo de margem quebra a rota de pedidos porque tudo está no mesmo escopo.

A estrutura abaixo é a convencional em projetos FastAPI:

```
app/
├── __init__.py
├── main.py            ← cria o FastAPI, inclui routers. NADA de lógica.
├── config.py          ← configuração (URL do banco, chaves)
├── banco.py           ← engine, SessionLocal, get_sessao
├── modelos.py         ← tabelas do SQLAlchemy (o que vai ao disco)
├── esquemas.py        ← modelos Pydantic (o que trafega na rede)
├── dependencias.py    ← funções compartilhadas via Depends
├── repositorio.py     ← acesso a dados. Só ele fala SQL.
├── servicos.py        ← regra de negócio. Não conhece HTTP.
└── rotas/
    ├── produtos.py
    └── pedidos.py
```

> 🎯 **A distinção que mais confunde: `modelos.py` vs `esquemas.py`.**
>
> | | `modelos.py` | `esquemas.py` |
> |---|---|---|
> | Biblioteca | SQLAlchemy | Pydantic |
> | Representa | Linha da tabela | Corpo da requisição/resposta |
> | Vive em | Disco | Rede |
> | Quem valida | O banco | A API |
>
> São **coisas diferentes** que por acaso têm campos parecidos. Não tente usar um no lugar do outro — é aí que os vazamentos de `custo` acontecem.

## 2. `Depends` — o coração do FastAPI

Antes de montar o projeto, entenda o mecanismo que sustenta tudo.

**Injeção de dependência** significa: em vez de a função *buscar* o que precisa, ela **declara** o que precisa e recebe pronto.

In [ ]:
from typing import Annotated
from fastapi import Depends, FastAPI

app = FastAPI()


def paginacao(pagina: int = 1, por_pagina: int = 20) -> dict:
    """Uma dependência é só uma função. O FastAPI a chama por você."""
    return {"pular": (pagina - 1) * por_pagina, "limite": por_pagina}


# 🎯 Annotated[Tipo, Depends(funcao)] — a sintaxe moderna
Paginacao = Annotated[dict, Depends(paginacao)]


@app.get("/produtos")
def listar_produtos(pag: Paginacao):
    return {"recurso": "produtos", **pag}


@app.get("/pedidos")
def listar_pedidos(pag: Paginacao):
    return {"recurso": "pedidos", **pag}


cliente = TestClient(app)
req(cliente, "GET", "/produtos?pagina=3&por_pagina=10")
req(cliente, "GET", "/pedidos")

> 💡 **Repare no que aconteceu de graça:**
>
> Os parâmetros `pagina` e `por_pagina` da dependência viraram **query params documentados** das duas rotas. O FastAPI inspeciona a assinatura da dependência e a costura na assinatura da rota.
>
> Escrevi a paginação **uma vez**. Se amanhã eu adicionar validação `ge=1`, as duas rotas ganham juntas.

In [ ]:
# Dependências ANINHADAS: uma depende da outra
from fastapi import Header, HTTPException


def obter_token(authorization: str = Header(default="")) -> str:
    """Nível 1: lê o cabeçalho."""
    if not authorization.startswith("Bearer "):
        raise HTTPException(401, "Token ausente ou malformado")
    return authorization.removeprefix("Bearer ")


def usuario_atual(token: Annotated[str, Depends(obter_token)]) -> dict:
    """Nível 2: usa o resultado do nível 1."""
    usuarios = {"tok-ana": {"nome": "Ana", "papel": "admin"},
                "tok-bruno": {"nome": "Bruno", "papel": "leitor"}}
    if token not in usuarios:
        raise HTTPException(401, "Token inválido")
    return usuarios[token]


def exigir_admin(usuario: Annotated[dict, Depends(usuario_atual)]) -> dict:
    """Nível 3: usa o resultado do nível 2."""
    if usuario["papel"] != "admin":
        raise HTTPException(403, f"{usuario['nome']} não é admin")
    return usuario


app = FastAPI()


@app.get("/meu-perfil")
def perfil(usuario: Annotated[dict, Depends(usuario_atual)]):
    return usuario


@app.delete("/produtos/{sku}")
def remover(sku: str, admin: Annotated[dict, Depends(exigir_admin)]):
    return {"removido": sku, "por": admin["nome"]}


cliente = TestClient(app)
req(cliente, "GET", "/meu-perfil")                                          # sem token
req(cliente, "GET", "/meu-perfil", headers={"Authorization": "Bearer tok-ana"})
req(cliente, "DELETE", "/produtos/NB-01", headers={"Authorization": "Bearer tok-bruno"})
req(cliente, "DELETE", "/produtos/NB-01", headers={"Authorization": "Bearer tok-ana"})

> 🎯 **O FastAPI montou o grafo sozinho.** `exigir_admin` → `usuario_atual` → `obter_token` → cabeçalho.
>
> Cada função tem uma responsabilidade e é testável isoladamente. E o `401` vs `403` sai certo: *"não sei quem você é"* vs *"sei quem você é e você não pode"*.

In [ ]:
# ⚠️ CACHE: dentro de UMA requisição, cada dependência roda UMA vez
chamadas = []


def cara() -> str:
    chamadas.append("!")
    return "resultado-caro"


def a(x: Annotated[str, Depends(cara)]): return x
def b(x: Annotated[str, Depends(cara)]): return x


app = FastAPI()


@app.get("/tres-usos")
def tres_usos(p: Annotated[str, Depends(cara)],
              q: Annotated[str, Depends(a)],
              r: Annotated[str, Depends(b)]):
    return {"chamadas_nesta_requisicao": len(chamadas)}


cliente = TestClient(app)
chamadas.clear()
req(cliente, "GET", "/tres-usos")

chamadas.clear()
cliente.get("/tres-usos")
cliente.get("/tres-usos")
print(f"Duas requisições → {len(chamadas)} chamada(s): o cache NÃO atravessa requisições.")

> 💡 **`use_cache=True` é o padrão** e é o que torna `get_sessao` seguro: dez dependências podem pedir a sessão e todas recebem **a mesma**, dentro daquela requisição.
>
> Para desligar: `Depends(funcao, use_cache=False)`.

## 3. 🔴 Sessão por requisição

Agora o ponto mais importante da aula.

In [ ]:
%%writefile aula_06_03/app/config.py
"""Configuração da aplicação."""
import os
from pathlib import Path

DIR_APP = Path(__file__).parent
DIR_RAIZ = DIR_APP.parent

# 🔑 A URL vem do ambiente. Em produção seria postgresql+psycopg://...
URL_BANCO = os.getenv("ATLAS_URL_BANCO", f"sqlite:///{DIR_RAIZ / 'atlas.db'}")

TITULO = "Atlas API"
VERSAO = "2.0.0"

In [ ]:
%%writefile aula_06_03/app/banco.py
"""Engine, fábrica de sessões e a dependência `get_sessao`."""
from collections.abc import Iterator

from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Session, sessionmaker

from app.config import URL_BANCO

# ── Engine: UM por processo. Ele gerencia o pool de conexões. ──
motor = create_engine(
    URL_BANCO,
    echo=False,
    pool_pre_ping=True,                       # 💡 testa a conexão antes de usar
    # `check_same_thread` é exclusivo do SQLite: o TestClient roda as rotas
    # em outra thread. Em PostgreSQL, remova.
    connect_args={"check_same_thread": False} if URL_BANCO.startswith("sqlite") else {},
)

# ── Fábrica de sessões ──
Sessao = sessionmaker(bind=motor, autoflush=False, expire_on_commit=False)


class Base(DeclarativeBase):
    """Base declarativa de todos os modelos."""


# ═══════════════════════════════════════════════════════════════
#  🔴 A dependência mais importante do projeto
# ═══════════════════════════════════════════════════════════════
def get_sessao() -> Iterator[Session]:
    """Abre UMA sessão para ESTA requisição e garante que ela fecha.

    O `yield` é o que faz a mágica: o FastAPI executa até o yield, entrega
    a sessão para a rota, e ao terminar a resposta volta para o `finally`.

    🔴 Sem o `finally: sessao.close()`, cada requisição vaza uma conexão
       do pool. O sintoma clássico: a API funciona por horas e de repente
       trava com `QueuePool limit of size 5 overflow 10 reached`.
    """
    sessao = Sessao()
    try:
        yield sessao
    finally:
        sessao.close()

> 🔴 **Por que uma sessão por requisição, e não uma global?**
>
> | Sessão global | Sessão por requisição |
> |---------------|-----------------------|
> | Compartilhada entre threads → **corrida** | Isolada |
> | Uma transação sem fim | Transação com escopo claro |
> | Erro em uma rota contamina as outras | Rollback afeta só uma |
> | Identity Map cresce para sempre → **vazamento** | Descartado ao fim |
>
> O `Session` do SQLAlchemy **não é thread-safe**. Uma sessão global numa API concorrente é um bug esperando o tráfego aumentar.
>
> 💭 **A analogia:** a sessão é um carrinho de compras, não a loja. O `Engine` (com o pool) é a loja — esse sim é global.

In [ ]:
%%writefile aula_06_03/app/modelos.py
"""Tabelas — o que vai ao disco. SQLAlchemy, não Pydantic."""
from datetime import datetime, timezone

from sqlalchemy import DateTime, ForeignKey, Numeric, String
from sqlalchemy.orm import Mapped, mapped_column, relationship

from app.banco import Base


def agora() -> datetime:
    return datetime.now(timezone.utc)


class Produto(Base):
    __tablename__ = "produtos"

    id: Mapped[int] = mapped_column(primary_key=True)
    sku: Mapped[str] = mapped_column(String(20), unique=True, index=True)
    nome: Mapped[str] = mapped_column(String(120))
    categoria: Mapped[str] = mapped_column(String(40), index=True)
    # 🔴 Numeric, não Float: dinheiro não tolera erro de ponto flutuante (M05)
    preco: Mapped[float] = mapped_column(Numeric(12, 2))
    custo: Mapped[float] = mapped_column(Numeric(12, 2))
    estoque: Mapped[int] = mapped_column(default=0)
    criado_em: Mapped[datetime] = mapped_column(DateTime, default=agora)

    itens: Mapped[list["ItemPedido"]] = relationship(back_populates="produto")


class Pedido(Base):
    __tablename__ = "pedidos"

    id: Mapped[int] = mapped_column(primary_key=True)
    cliente_email: Mapped[str] = mapped_column(String(120), index=True)
    canal: Mapped[str] = mapped_column(String(20))
    status: Mapped[str] = mapped_column(String(20), default="pendente", index=True)
    criado_em: Mapped[datetime] = mapped_column(DateTime, default=agora)

    itens: Mapped[list["ItemPedido"]] = relationship(
        back_populates="pedido", cascade="all, delete-orphan")


class ItemPedido(Base):
    __tablename__ = "itens_pedido"

    id: Mapped[int] = mapped_column(primary_key=True)
    pedido_id: Mapped[int] = mapped_column(ForeignKey("pedidos.id"), index=True)
    produto_id: Mapped[int] = mapped_column(ForeignKey("produtos.id"), index=True)
    quantidade: Mapped[int]
    preco_unitario: Mapped[float] = mapped_column(Numeric(12, 2))

    pedido: Mapped["Pedido"] = relationship(back_populates="itens")
    produto: Mapped["Produto"] = relationship(back_populates="itens")

In [ ]:
%%writefile aula_06_03/app/esquemas.py
"""Contratos de rede — o que trafega. Pydantic, não SQLAlchemy."""
from datetime import datetime

from pydantic import BaseModel, ConfigDict, Field, field_validator


# ═══════════ Produto ═══════════
class ProdutoBase(BaseModel):
    nome: str = Field(min_length=3, max_length=120)
    categoria: str = Field(min_length=2, max_length=40)
    preco: float = Field(gt=0)


class ProdutoCriar(ProdutoBase):
    sku: str = Field(min_length=5, max_length=20, pattern=r"^[A-Z]{2}-[A-Z0-9-]+$")
    custo: float = Field(ge=0)
    estoque: int = Field(default=0, ge=0)

    @field_validator("sku", mode="before")
    @classmethod
    def maiusculo(cls, v):
        return v.strip().upper() if isinstance(v, str) else v


class ProdutoAtualizar(BaseModel):
    nome: str | None = Field(default=None, min_length=3, max_length=120)
    categoria: str | None = Field(default=None, min_length=2)
    preco: float | None = Field(default=None, gt=0)
    estoque: int | None = Field(default=None, ge=0)


class ProdutoResposta(ProdutoBase):
    # 🔑 from_attributes: permite construir a partir de um OBJETO
    #    (o modelo do SQLAlchemy), não só de um dicionário.
    model_config = ConfigDict(from_attributes=True)

    id: int
    sku: str
    estoque: int
    criado_em: datetime
    # 🔒 `custo` NÃO aparece aqui. É de propósito.


class ListaProdutos(BaseModel):
    total: int
    itens: list[ProdutoResposta]


# ═══════════ Pedido ═══════════
class ItemEntrada(BaseModel):
    sku: str = Field(min_length=5)
    quantidade: int = Field(gt=0, le=1000)


class PedidoCriar(BaseModel):
    cliente_email: str = Field(pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    canal: str = Field(pattern="^(site|app|marketplace)$")
    itens: list[ItemEntrada] = Field(min_length=1, max_length=50)


class ItemResposta(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    sku: str
    nome: str
    quantidade: int
    preco_unitario: float
    subtotal: float


class PedidoResposta(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    cliente_email: str
    canal: str
    status: str
    criado_em: datetime
    itens: list[ItemResposta]
    total: float


class Erro(BaseModel):
    codigo: str
    mensagem: str

> 🔑 **`from_attributes=True`** é o que permite `ProdutoResposta.model_validate(objeto_sqlalchemy)`.
>
> Sem ele, o Pydantic só aceitaria dicionários. Com ele, ele lê os atributos do objeto — e o `response_model` do FastAPI faz isso automaticamente quando você retorna um modelo do SQLAlchemy.
>
> ⚠️ **Na versão 1 do Pydantic isso se chamava `orm_mode`.** Você vai encontrar tutoriais antigos usando esse nome.

## 4. Repositório e serviço

O repositório é o **único** lugar que fala SQLAlchemy. O serviço é o único que decide regra de negócio.

In [ ]:
%%writefile aula_06_03/app/repositorio.py
"""Acesso a dados. Só este módulo conhece SQLAlchemy.

Nenhuma função aqui faz `commit`. Quem controla a transação é a camada
de cima — porque só ela sabe onde a unidade de trabalho começa e termina.
"""
from sqlalchemy import func, select
from sqlalchemy.orm import Session, selectinload

from app.modelos import ItemPedido, Pedido, Produto


# ═══════════ Produtos ═══════════
def buscar_produto_por_sku(sessao: Session, sku: str) -> Produto | None:
    return sessao.scalar(select(Produto).where(Produto.sku == sku))


def buscar_produto(sessao: Session, produto_id: int) -> Produto | None:
    return sessao.get(Produto, produto_id)


def listar_produtos(sessao: Session, categoria: str | None = None,
                    pular: int = 0, limite: int = 20) -> list[Produto]:
    consulta = select(Produto)
    if categoria:
        consulta = consulta.where(Produto.categoria == categoria)
    consulta = consulta.order_by(Produto.sku).offset(pular).limit(limite)
    return list(sessao.scalars(consulta))


def contar_produtos(sessao: Session, categoria: str | None = None) -> int:
    consulta = select(func.count()).select_from(Produto)
    if categoria:
        consulta = consulta.where(Produto.categoria == categoria)
    return sessao.scalar(consulta) or 0


def inserir_produto(sessao: Session, **campos) -> Produto:
    produto = Produto(**campos)
    sessao.add(produto)
    sessao.flush()          # 💡 flush, não commit: gera o id sem encerrar
    return produto


def remover_produto(sessao: Session, produto: Produto) -> None:
    sessao.delete(produto)


# ═══════════ Pedidos ═══════════
def buscar_pedido(sessao: Session, pedido_id: int) -> Pedido | None:
    """🎯 selectinload evita o N+1 ao carregar itens e produtos junto."""
    return sessao.scalar(
        select(Pedido)
        .where(Pedido.id == pedido_id)
        .options(selectinload(Pedido.itens).selectinload(ItemPedido.produto))
    )


def listar_pedidos(sessao: Session, status: str | None = None,
                   pular: int = 0, limite: int = 20) -> list[Pedido]:
    consulta = select(Pedido).options(
        selectinload(Pedido.itens).selectinload(ItemPedido.produto))
    if status:
        consulta = consulta.where(Pedido.status == status)
    consulta = consulta.order_by(Pedido.id.desc()).offset(pular).limit(limite)
    return list(sessao.scalars(consulta))


def inserir_pedido(sessao: Session, pedido: Pedido) -> Pedido:
    sessao.add(pedido)
    sessao.flush()
    return pedido

In [ ]:
%%writefile aula_06_03/app/servicos.py
"""Regra de negócio. Não importa NADA de fastapi.

Se um dia o Atlas ganhar um worker de fila ou um comando de CLI, esta
camada é reaproveitada inteira. É por isso que ela levanta exceções de
domínio em vez de HTTPException.
"""
from decimal import Decimal

from sqlalchemy.orm import Session

from app import repositorio
from app.modelos import ItemPedido, Pedido


# ═══════════ Exceções de domínio ═══════════
class AtlasError(Exception):
    codigo = "erro"

    def __init__(self, mensagem: str):
        self.mensagem = mensagem
        super().__init__(mensagem)


class NaoEncontrado(AtlasError):
    codigo = "nao_encontrado"


class JaExiste(AtlasError):
    codigo = "ja_existe"


class EstoqueInsuficiente(AtlasError):
    codigo = "estoque_insuficiente"


class RegraViolada(AtlasError):
    codigo = "regra_violada"


# ═══════════ Produtos ═══════════
def criar_produto(sessao: Session, dados) -> object:
    if repositorio.buscar_produto_por_sku(sessao, dados.sku):
        raise JaExiste(f"SKU {dados.sku} já cadastrado")
    if dados.preco < dados.custo:
        raise RegraViolada(f"preço {dados.preco} abaixo do custo {dados.custo}")
    produto = repositorio.inserir_produto(sessao, **dados.model_dump())
    sessao.commit()                      # 🔑 o SERVIÇO decide onde commitar
    return produto


def obter_produto(sessao: Session, sku: str) -> object:
    produto = repositorio.buscar_produto_por_sku(sessao, sku)
    if produto is None:
        raise NaoEncontrado(f"Produto {sku} não encontrado")
    return produto


def atualizar_produto(sessao: Session, sku: str, dados) -> object:
    produto = obter_produto(sessao, sku)
    alteracoes = dados.model_dump(exclude_unset=True)
    if not alteracoes:
        raise RegraViolada("nenhum campo enviado")
    for campo, valor in alteracoes.items():
        setattr(produto, campo, valor)
    if Decimal(str(produto.preco)) < Decimal(str(produto.custo)):
        sessao.rollback()                # 🔴 desfaz antes de sair
        raise RegraViolada("a alteração deixaria o preço abaixo do custo")
    sessao.commit()
    return produto


def remover_produto(sessao: Session, sku: str) -> None:
    produto = obter_produto(sessao, sku)
    repositorio.remover_produto(sessao, produto)
    sessao.commit()


# ═══════════ Pedidos ═══════════
def criar_pedido(sessao: Session, dados) -> Pedido:
    """🎯 Ou o pedido inteiro entra, ou nada entra."""
    pedido = Pedido(cliente_email=dados.cliente_email, canal=dados.canal)

    problemas = []
    for item in dados.itens:
        produto = repositorio.buscar_produto_por_sku(sessao, item.sku)
        if produto is None:
            problemas.append(f"{item.sku}: não existe")
            continue
        if produto.estoque < item.quantidade:
            problemas.append(
                f"{item.sku}: pedido {item.quantidade}, disponível {produto.estoque}")
            continue
        produto.estoque -= item.quantidade
        pedido.itens.append(ItemPedido(
            produto_id=produto.id, quantidade=item.quantidade,
            preco_unitario=produto.preco))

    if problemas:
        sessao.rollback()                # 🔴 nada do que fizemos acima persiste
        raise EstoqueInsuficiente("; ".join(problemas))

    repositorio.inserir_pedido(sessao, pedido)
    sessao.commit()
    return pedido


def obter_pedido(sessao: Session, pedido_id: int) -> Pedido:
    pedido = repositorio.buscar_pedido(sessao, pedido_id)
    if pedido is None:
        raise NaoEncontrado(f"Pedido {pedido_id} não encontrado")
    return pedido


def montar_resposta_pedido(pedido: Pedido) -> dict:
    """Traduz o objeto do banco para o formato da resposta."""
    itens = [{
        "sku": i.produto.sku, "nome": i.produto.nome,
        "quantidade": i.quantidade, "preco_unitario": float(i.preco_unitario),
        "subtotal": round(i.quantidade * float(i.preco_unitario), 2),
    } for i in pedido.itens]
    return {
        "id": pedido.id, "cliente_email": pedido.cliente_email,
        "canal": pedido.canal, "status": pedido.status,
        "criado_em": pedido.criado_em, "itens": itens,
        "total": round(sum(i["subtotal"] for i in itens), 2),
    }

> 🎯 **Quem faz `commit`?**
>
> | Camada | Responsabilidade |
> |--------|------------------|
> | Repositório | `add`, `select`, `delete`, `flush` — **nunca** `commit` |
> | **Serviço** | Decide onde a transação começa e termina → **`commit` / `rollback`** |
> | Rota | Nem sabe que existe transação |
>
> **Por quê?** Só o serviço sabe que "criar pedido" significa *baixar estoque de 3 produtos E inserir 4 linhas*, tudo ou nada. Se o repositório commitasse a cada `add`, essa atomicidade seria impossível.
>
> 💡 **`flush` vs `commit`:** o `flush` manda o SQL para o banco e obtém o `id` gerado, mas **não encerra a transação** — ainda dá para desfazer.

## 5. Rotas com `APIRouter`

In [ ]:
%%writefile aula_06_03/app/dependencias.py
"""Dependências compartilhadas entre routers."""
from typing import Annotated

from fastapi import Depends, Query
from sqlalchemy.orm import Session

from app.banco import get_sessao

# 🔑 Aliases: escreve-se uma vez, usa-se em todas as rotas.
SessaoDep = Annotated[Session, Depends(get_sessao)]


def paginacao(pagina: Annotated[int, Query(ge=1)] = 1,
              por_pagina: Annotated[int, Query(ge=1, le=100)] = 20) -> dict:
    return {"pular": (pagina - 1) * por_pagina, "limite": por_pagina}


PaginacaoDep = Annotated[dict, Depends(paginacao)]

In [ ]:
%%writefile aula_06_03/app/rotas/produtos.py
"""Rotas de produto. Só traduz HTTP ↔ serviço."""
from fastapi import APIRouter, status

from app import esquemas, repositorio, servicos
from app.dependencias import PaginacaoDep, SessaoDep

roteador = APIRouter(prefix="/produtos", tags=["Produtos"])


@roteador.post("", response_model=esquemas.ProdutoResposta,
               status_code=status.HTTP_201_CREATED,
               responses={409: {"model": esquemas.Erro}})
def criar(dados: esquemas.ProdutoCriar, sessao: SessaoDep):
    return servicos.criar_produto(sessao, dados)


@roteador.get("", response_model=esquemas.ListaProdutos)
def listar(sessao: SessaoDep, pag: PaginacaoDep, categoria: str | None = None):
    return {
        "total": repositorio.contar_produtos(sessao, categoria),
        "itens": repositorio.listar_produtos(sessao, categoria, **pag),
    }


@roteador.get("/{sku}", response_model=esquemas.ProdutoResposta,
              responses={404: {"model": esquemas.Erro}})
def obter(sku: str, sessao: SessaoDep):
    return servicos.obter_produto(sessao, sku)


@roteador.patch("/{sku}", response_model=esquemas.ProdutoResposta,
                responses={404: {"model": esquemas.Erro}})
def atualizar(sku: str, dados: esquemas.ProdutoAtualizar, sessao: SessaoDep):
    return servicos.atualizar_produto(sessao, sku, dados)


@roteador.delete("/{sku}", status_code=status.HTTP_204_NO_CONTENT)
def remover(sku: str, sessao: SessaoDep):
    servicos.remover_produto(sessao, sku)

In [ ]:
%%writefile aula_06_03/app/rotas/pedidos.py
"""Rotas de pedido."""
from fastapi import APIRouter, status

from app import esquemas, repositorio, servicos
from app.dependencias import PaginacaoDep, SessaoDep

roteador = APIRouter(prefix="/pedidos", tags=["Pedidos"])


@roteador.post("", response_model=esquemas.PedidoResposta,
               status_code=status.HTTP_201_CREATED,
               responses={409: {"model": esquemas.Erro}})
def criar(dados: esquemas.PedidoCriar, sessao: SessaoDep):
    pedido = servicos.criar_pedido(sessao, dados)
    return servicos.montar_resposta_pedido(pedido)


@roteador.get("", response_model=list[esquemas.PedidoResposta])
def listar(sessao: SessaoDep, pag: PaginacaoDep, status_: str | None = None):
    pedidos = repositorio.listar_pedidos(sessao, status_, **pag)
    return [servicos.montar_resposta_pedido(p) for p in pedidos]


@roteador.get("/{pedido_id}", response_model=esquemas.PedidoResposta,
              responses={404: {"model": esquemas.Erro}})
def obter(pedido_id: int, sessao: SessaoDep):
    return servicos.montar_resposta_pedido(servicos.obter_pedido(sessao, pedido_id))

> 💡 **Repare no tamanho das rotas.** Nenhuma passa de 4 linhas. Elas não fazem nada além de:
>
> 1. Declarar o que recebem (o FastAPI valida)
> 2. Chamar o serviço
> 3. Declarar o que devolvem (o FastAPI filtra)
>
> **Isso é o objetivo.** A rota é uma casca fina. Toda a lógica que você vai querer testar sem HTTP está em `servicos.py`.

## 6. `lifespan` e o `main.py`

In [ ]:
%%writefile aula_06_03/app/main.py
"""Ponto de entrada. Monta a aplicação — e nada mais."""
from contextlib import asynccontextmanager

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

from app import servicos
from app.banco import Base, motor
from app.config import TITULO, VERSAO
from app.rotas import pedidos, produtos


# ═══════════ Ciclo de vida ═══════════
@asynccontextmanager
async def ciclo_de_vida(aplicacao: FastAPI):
    """Roda UMA vez ao subir e UMA vez ao descer.

    Antes do `yield`: abrir recursos (criar tabelas, aquecer cache,
    conectar no Redis). Depois: fechar tudo com educação.

    ⚠️ `create_all` é aceitável em aula e em teste. Em produção quem
       cria e altera tabela é o Alembic (M05) — o `create_all` não sabe
       fazer migração, só cria o que não existe.
    """
    Base.metadata.create_all(motor)
    print("   [lifespan] tabelas prontas, API no ar")
    yield
    motor.dispose()
    print("   [lifespan] conexões devolvidas, API desligada")


app = FastAPI(title=TITULO, version=VERSAO, lifespan=ciclo_de_vida)


# ═══════════ Tradução domínio → HTTP ═══════════
_STATUS = {
    servicos.NaoEncontrado: 404,
    servicos.JaExiste: 409,
    servicos.EstoqueInsuficiente: 409,
    servicos.RegraViolada: 422,
}


@app.exception_handler(servicos.AtlasError)
def tratar_atlas(requisicao: Request, erro: servicos.AtlasError):
    codigo_http = _STATUS.get(type(erro), 400)
    return JSONResponse(status_code=codigo_http,
                        content={"codigo": erro.codigo, "mensagem": erro.mensagem})


# ═══════════ Routers ═══════════
app.include_router(produtos.roteador)
app.include_router(pedidos.roteador)


@app.get("/saude", tags=["Infra"])
def saude():
    return {"status": "ok", "versao": VERSAO}

In [ ]:
%%writefile aula_06_03/app/__init__.py
"""Pacote da API do Atlas."""

In [ ]:
%%writefile aula_06_03/app/rotas/__init__.py
"""Routers da API."""

In [ ]:
print("Estrutura criada:\n")
arvore(BASE)

## 7. A API rodando

In [ ]:
recarregar("app")                 # limpa o cache de import
from app.main import app          # noqa: E402

# 🔑 O `with` dispara o lifespan: entrada = startup, saída = shutdown
with TestClient(app) as cliente:
    req(cliente, "GET", "/saude")

    req(cliente, "POST", "/produtos", json={
        "sku": "nb-dell-15", "nome": "Notebook Dell Inspiron 15",
        "categoria": "Notebooks", "preco": 2599.90, "custo": 2120.00,
        "estoque": 14})

> 🎯 **Repare que a resposta não tem `custo`.**
>
> O serviço devolveu o objeto `Produto` inteiro — com `custo`, com `criado_em`, com tudo. O `response_model=ProdutoResposta` filtrou.
>
> Essa é a diferença entre "esquecer de esconder" e "não ter como vazar".

In [ ]:
# Sessão persistente para o resto da aula
cliente = TestClient(app)
cliente.__enter__()               # dispara o startup

CATALOGO = [
    {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide", "categoria": "Monitores",
     "preco": 1199.00, "custo": 920.00, "estoque": 31},
    {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170", "categoria": "Periféricos",
     "preco": 89.90, "custo": 52.00, "estoque": 120},
    {"sku": "PE-RED-K552", "nome": "Teclado Redragon K552", "categoria": "Periféricos",
     "preco": 249.90, "custo": 168.00, "estoque": 8},
    {"sku": "AR-KING-1TB", "nome": "SSD Kingston NV2 1TB", "categoria": "Armazenamento",
     "preco": 429.00, "custo": 305.00, "estoque": 47},
]
for p in CATALOGO:
    cliente.post("/produtos", json=p)

req(cliente, "GET", "/produtos?por_pagina=3")

In [ ]:
print("── filtro + paginação ──")
req(cliente, "GET", "/produtos?categoria=Periféricos", mostrar_corpo=False)
r = cliente.get("/produtos?categoria=Periféricos")
for item in r.json()["itens"]:
    print(f"   {item['sku']:<14} {item['nome']:<28} R$ {item['preco']:>8.2f}")

print("\n── SKU duplicado (regra do serviço) ──")
req(cliente, "POST", "/produtos", json={
    "sku": "MO-LG-24UW", "nome": "Duplicado", "categoria": "Monitores",
    "preco": 100, "custo": 50})

print("── preço abaixo do custo ──")
req(cliente, "POST", "/produtos", json={
    "sku": "XX-TESTE-1", "nome": "Teste", "categoria": "Testes",
    "preco": 50, "custo": 100})

print("── inexistente ──")
req(cliente, "GET", "/produtos/NAO-EXISTE")

## 8. 🔴 A transação em ação

Aqui está o motivo de todo o trabalho de arquitetura: **atomicidade**.

In [ ]:
print("Estoque ANTES:")
for sku in ["NB-DELL-15", "PE-RED-K552"]:
    print(f"   {sku:<14} {cliente.get(f'/produtos/{sku}').json()['estoque']}")

print("\n── pedido válido ──")
req(cliente, "POST", "/pedidos", json={
    "cliente_email": "maria@aurora.com.br", "canal": "site",
    "itens": [{"sku": "NB-DELL-15", "quantidade": 2},
              {"sku": "PE-RED-K552", "quantidade": 1}]})

print("Estoque DEPOIS:")
for sku in ["NB-DELL-15", "PE-RED-K552"]:
    print(f"   {sku:<14} {cliente.get(f'/produtos/{sku}').json()['estoque']}")

In [ ]:
# 🔴 O TESTE QUE IMPORTA: o primeiro item cabe, o segundo não.
print("Estoque ANTES:")
for sku in ["AR-KING-1TB", "PE-RED-K552"]:
    print(f"   {sku:<14} {cliente.get(f'/produtos/{sku}').json()['estoque']}")

print("\n── pedido misto: item 1 disponível, item 2 estourando ──")
req(cliente, "POST", "/pedidos", json={
    "cliente_email": "joao@aurora.com.br", "canal": "app",
    "itens": [{"sku": "AR-KING-1TB", "quantidade": 5},
              {"sku": "PE-RED-K552", "quantidade": 999}]})

print("Estoque DEPOIS:")
for sku in ["AR-KING-1TB", "PE-RED-K552"]:
    print(f"   {sku:<14} {cliente.get(f'/produtos/{sku}').json()['estoque']}")

> 🔴 **O estoque do SSD não mudou.**
>
> O serviço já tinha feito `produto.estoque -= 5` na memória. Mas ao encontrar o segundo problema chamou `sessao.rollback()` — e o SQLAlchemy descartou a alteração.
>
> **Sem transação, esse pedido teria "sumido" com 5 SSDs.** O cliente recebe um erro, o estoque fica errado, e ninguém descobre até a contagem do fim do mês.
>
> 💭 Note que a rota **não sabe nada disso**. Ela chamou `servicos.criar_pedido` e o handler traduziu a exceção. A atomicidade mora inteiramente no serviço.

In [ ]:
# Pedido com SKU inexistente + acumulação de problemas
req(cliente, "POST", "/pedidos", json={
    "cliente_email": "ana@aurora.com.br", "canal": "marketplace",
    "itens": [{"sku": "ZZ-FANTASMA", "quantidade": 1},
              {"sku": "PE-RED-K552", "quantidade": 500}]})

print("── listagem de pedidos ──")
r = cliente.get("/pedidos")
for p in r.json():
    itens = ", ".join(f"{i['quantidade']}× {i['sku']}" for i in p["itens"])
    print(f"   #{p['id']}  {p['cliente_email']:<24} R$ {p['total']:>9.2f}  ({itens})")

## 9. Sobrescrever dependências — teste de verdade

In [ ]:
# 🎯 dependency_overrides: troca o banco real por um de teste
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

from app.banco import Base, get_sessao

# ⚠️ ARMADILHA: SQLite em memória é POR CONEXÃO. O TestClient roda as
#    rotas em outra thread → nova conexão → banco vazio → "no such table".
#    `StaticPool` força TODAS as conexões a serem a MESMA. Sem ele, este
#    bloco quebra de um jeito confuso.
motor_teste = create_engine("sqlite:///:memory:",
                            connect_args={"check_same_thread": False},
                            poolclass=StaticPool)
SessaoTeste = sessionmaker(bind=motor_teste, expire_on_commit=False)
Base.metadata.create_all(motor_teste)


def get_sessao_teste():
    sessao = SessaoTeste()
    try:
        yield sessao
    finally:
        sessao.close()


app.dependency_overrides[get_sessao] = get_sessao_teste

cliente_teste = TestClient(app)
print("Banco de teste — vazio, isolado do de produção:")
req(cliente_teste, "GET", "/produtos")

req(cliente_teste, "POST", "/produtos", json={
    "sku": "TS-UNICO-01", "nome": "Produto de teste", "categoria": "Testes",
    "preco": 10.0, "custo": 5.0, "estoque": 1}, mostrar_corpo=False)

print(f"No banco de TESTE      : {cliente_teste.get('/produtos').json()['total']} produto(s)")
app.dependency_overrides.clear()
print(f"No banco de PRODUÇÃO   : {cliente.get('/produtos').json()['total']} produto(s)")

> 🎯 **`dependency_overrides` é a razão prática de usar `Depends` para tudo.**
>
> Como a rota nunca chamou `Sessao()` diretamente — ela **declarou** que precisa de uma sessão — dá para trocar a implementação de fora, sem tocar em uma linha de código de rota.
>
> O mesmo vale para autenticação (`usuario_atual`), clientes HTTP externos, envio de e-mail. **Se algo é `Depends`, é substituível no teste.**
>
> Você vai usar isso a fundo no Módulo 07.

> ⚠️ **A armadilha do `sqlite:///:memory:` que quase todo mundo pega.**
>
> Um banco SQLite em memória pertence à **conexão**, não ao processo. O `TestClient` executa rotas `def` (síncronas) numa *thread pool* — e o pool padrão do SQLAlchemy para `:memory:` abre uma conexão por thread.
>
> Resultado: você cria as tabelas na thread principal e a rota, em outra thread, encontra um banco **vazio**. O erro é `no such table: produtos`, que parece um problema de modelo e não é.
>
> **A correção é `poolclass=StaticPool`** — uma conexão só, compartilhada. Guarde essa: ela aparece em quase todo projeto FastAPI + SQLite que tem testes.

## 🔧 Prática guiada — o mapa da API

In [ ]:
especificacao = cliente.get("/openapi.json").json()

print(f"{especificacao['info']['title']} v{especificacao['info']['version']}\n")
print(f"{'MÉTODO':<8} {'ROTA':<24} {'TAG':<12} RESPOSTAS")
print("─" * 66)
for caminho, metodos in sorted(especificacao["paths"].items()):
    for metodo, detalhe in metodos.items():
        tag = (detalhe.get("tags") or ["—"])[0]
        codigos = ",".join(sorted(detalhe["responses"]))
        print(f"{metodo.upper():<8} {caminho:<24} {tag:<12} {codigos}")

print(f"\nEsquemas gerados: {len(especificacao['components']['schemas'])}")
for nome in sorted(especificacao["components"]["schemas"]):
    print(f"   · {nome}")

In [ ]:
# Prova de que `custo` não vaza em NENHUMA resposta documentada
esquemas_saida = ["ProdutoResposta", "ListaProdutos", "PedidoResposta"]
for nome in esquemas_saida:
    campos = list(especificacao["components"]["schemas"][nome].get("properties", {}))
    marca = "🔴 VAZOU" if "custo" in campos else "✅"
    print(f"{marca} {nome:<18} {campos}")

In [ ]:
# Encerra o cliente com educação (dispara o shutdown do lifespan)
cliente.__exit__(None, None, None)

## 📝 Exercícios

**E1.** Desenhe no papel (ou em markdown) o fluxo completo de `POST /pedidos`: quais módulos são tocados, em que ordem, e onde a transação abre e fecha.

**E2.** Crie uma dependência `filtro_data(inicio, fim)` que valide que `inicio <= fim` e devolva um dicionário. Use em duas rotas diferentes.

**E3.** Prove o cache de dependências: crie uma que imprima algo e use-a três vezes na mesma rota. Depois desligue com `use_cache=False` e compare.

**E4.** Adicione um `APIRouter` de clientes com CRUD completo, seguindo a mesma estrutura de camadas.

**E5.** 🔴 Escreva uma versão de `get_sessao` **sem** o `finally: close()`. Faça 30 requisições com `pool_size=2, max_overflow=0` e mostre o erro que aparece.

**E6.** Mova o `commit` do serviço para o repositório e mostre, com o teste do pedido misto, que a atomicidade se perde.

**E7.** Adicione um router `/relatorios` com uma rota de faturamento por categoria, usando `func.sum` e `group_by` no repositório.

**E8.** Implemente `PUT /produtos/{sku}` (substituição total) e explique em comentários quando usar `PUT` e quando usar `PATCH`.

**E9.** Use `dependency_overrides` para criar uma fixture de teste que reinicia o banco a cada requisição.

**E10.** Adicione ao `lifespan` a criação de um índice e a impressão de quantos produtos existem ao subir.

**E11.** Crie uma dependência `ordenacao(campo, direcao)` que valide o campo contra uma **lista branca** e explique por que isso é obrigatório.

**E12.** Meça o N+1: liste 20 pedidos com e sem `selectinload` usando `echo=True` no engine e conte as consultas.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

## 📋 Cola de referência

```python
# ═══ Estrutura ═══
# main.py       cria o app, inclui routers, handlers de exceção
# config.py     configuração (via ambiente)
# banco.py      motor, Sessao, Base, get_sessao
# modelos.py    SQLAlchemy — o que vai ao disco
# esquemas.py   Pydantic  — o que trafega na rede
# repositorio.py  só ele fala SQL. NUNCA commita.
# servicos.py     regra de negócio. Commita. Não importa fastapi.
# rotas/          cascas finas: recebe → chama serviço → devolve

# ═══ Sessão por requisição 🔴 ═══
def get_sessao() -> Iterator[Session]:
    sessao = Sessao()
    try:
        yield sessao
    finally:
        sessao.close()          # 🔴 sem isto, o pool esgota

SessaoDep = Annotated[Session, Depends(get_sessao)]

# ═══ Depends ═══
Dep = Annotated[Tipo, Depends(funcao)]     # sintaxe moderna
Depends(funcao, use_cache=False)           # desliga o cache por requisição
@roteador.get("/x", dependencies=[Depends(exigir_admin)])   # sem usar o valor
app.dependency_overrides[real] = falso     # 🎯 teste

# ═══ APIRouter ═══
roteador = APIRouter(prefix="/produtos", tags=["Produtos"])
app.include_router(roteador)
app.include_router(outro, prefix="/v1", dependencies=[Depends(auth)])

# ═══ lifespan ═══
@asynccontextmanager
async def ciclo(app: FastAPI):
    ...            # startup
    yield
    ...            # shutdown

app = FastAPI(lifespan=ciclo)
with TestClient(app) as c:   # o `with` dispara startup/shutdown
    ...

# ═══ Pydantic ↔ SQLAlchemy ═══
class Resposta(BaseModel):
    model_config = ConfigDict(from_attributes=True)   # lê ATRIBUTOS

# ═══ Transação ═══
sessao.flush()      # manda o SQL, gera ids, NÃO encerra
sessao.commit()     # confirma  — só no serviço
sessao.rollback()   # desfaz tudo desde o último commit
```

## ✅ Checklist de saída

- [ ] Sei a diferença entre `modelos.py` (SQLAlchemy) e `esquemas.py` (Pydantic)
- [ ] Uso `Annotated[Tipo, Depends(f)]` e entendo o grafo de dependências
- [ ] Sei que dependências têm cache **por requisição**
- [ ] 🔴 **Escrevo `get_sessao` com `try/finally: close()`**
- [ ] Sei por que uma sessão global é um bug
- [ ] Uso `from_attributes=True` nos esquemas de resposta
- [ ] O repositório **nunca** commita
- [ ] O serviço decide `commit` e `rollback`
- [ ] O serviço **não importa `fastapi`**
- [ ] Minhas rotas têm menos de 5 linhas
- [ ] Organizo rotas com `APIRouter` + `prefix` + `tags`
- [ ] Uso `lifespan` para abrir e fechar recursos
- [ ] Sei que `create_all` não substitui o Alembic
- [ ] Uso `dependency_overrides` para testar
- [ ] 🔒 Meus esquemas de resposta não expõem `custo`

---

### ➡️ Próxima aula

**`06_04_Seguranca_e_Filtros.ipynb`** — Configuração por ambiente, autenticação com JWT, autorização por papel, middlewares e CORS. Fechar a porta que acabamos de abrir.